# Embedding figures

Draws the embedding figures from the saved results; nothing is computed here. Run `embed.py` and `evaluate.py` first (see README).
Paper figures are written to `figures/`; the \"other methods\" figure of each dataset is only a sanity check, shown here and not saved. The number in the corner of each panel is the triplet accuracy of that very embedding, and the
embeddings are those of seed `SEED`.

In [ ]:
import sys
sys.path.insert(0, "..")  # make the modules in code/ importable

from functools import lru_cache

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from sklearn.isotonic import IsotonicRegression

import paths
import results_io as rio
from datasets import DATASETS
from display import METHOD_LABELS, method_for
from methods import METHODS
from plan import is_planned, main_jobs

paths.FIGURES.mkdir(exist_ok=True)
if paths.STYLE.exists():
    plt.style.use(paths.STYLE)
else:
    print(f"{paths.STYLE} not found: using matplotlib's default style. Copy your mpl_style.txt there.")

SEED = 0  # whose embeddings to draw

## Helpers

In [ ]:
@lru_cache(maxsize=1)
def load_data(dataset):
    return DATASETS[dataset]["load"]()  # X, y, precomputed


def _job(dataset, method):
    # deterministic methods only have seed 0; main_jobs knows
    return main_jobs([dataset], [method], [SEED])[0]


def load_Z(dataset, method):
    return rio.load_embedding(_job(dataset, method).emb_path)


def load_metric(dataset, method, name):
    return rio.load_metrics(_job(dataset, method).metrics_path)[name]


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(paths.FIGURES / f"{name}.{ext}", dpi=300)


def add_shepard(ax, dataset, Z, n_points, xlim, ylim, yticks):
    """Shepard diagram (embedding vs. normalized original distances) with the isotonic fit."""
    X, _, precomputed = load_data(dataset)
    ind = np.random.default_rng(seed=42).choice(X.shape[0], size=n_points, replace=False)
    d1 = X[np.ix_(ind, ind)] if precomputed else squareform(pdist(X[ind]))
    d2 = squareform(pdist(Z[ind]))
    d1 = d1 / d1.mean()

    ir = IsotonicRegression(out_of_bounds="clip").fit(d1.ravel(), d2.ravel())

    ax.scatter(d1.ravel(), d2.ravel(), s=1, ec="none", rasterized=True)
    ax.set_xlabel("Norm. high-dim. distances")
    ax.set_ylabel("NMDS embedding distances")
    xgrid = np.arange(0, 2.5, 0.01)
    ax.plot(xgrid, ir.predict(xgrid), "k--")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_yticks(yticks)
    ax.spines[["right", "top"]].set_visible(False)


def embedding_figure(
    dataset, name, s, cmap=None, shepard_n=500, shepard_xlim=(0, 2), shepard_ylim=(0, 3),
    shepard_yticks=(0, 1, 2, 3), text_fontsize=None, circle=False,
):
    """Classical / metric / non-metric MDS (a-c) and the Shepard diagram of the non-metric one (d)."""
    X, y, precomputed = load_data(dataset)
    rows = [("Classical", "Classical MDS"), ("Metric", "Metric MDS"), ("Non-metric", "Non-metric MDS")]

    fig, axs = plt.subplots(ncols=4, figsize=(5.5, 1.5), layout="constrained")
    for ax, (row, title) in zip(axs, rows):
        method = method_for(row, dataset)
        Z = load_Z(dataset, method)
        ax.set_aspect("equal", adjustable="datalim")
        ax.scatter(Z[:, 0], Z[:, 1], s=s, ec="none", c=y, cmap=cmap, rasterized=True)
        ax.set_axis_off()
        ax.set_title(title)
        ax.text(.99, .01, f"{load_metric(dataset, method, 'triplet_acc'):.2f}", ha="right",
                transform=ax.transAxes, fontsize=text_fontsize)
        if circle and row == "Metric":  # circle of radius (pi/4) x mean original distance
            d = squareform(X, checks=False) if precomputed else pdist(X)
            ax.add_patch(plt.Circle((0, 0), d[d > 0].mean() * np.pi / 4, facecolor="none",
                                    edgecolor="k", linestyle="--", lw=.6, zorder=0))

    axs[3].set_visible(False)
    axs[3] = plt.axes([.81, .2, .14, .6])
    add_shepard(axs[3], dataset, Z, shepard_n, shepard_xlim, shepard_ylim, shepard_yticks)

    for x, letter in zip([.01, .25, .50, .75], "abcd"):
        fig.text(x, .93, letter, fontweight="bold", fontsize=8)
    save(fig, name)
    return fig


def other_methods_figure(dataset, s, cmap=None):
    """Sanity check, not a paper figure: shown here and not saved. Every method that is run on
    this dataset and not shown in the main figure (t-SNE, UMAP, PaCMAP, the SGD versions where
    the main figure shows SMACOF, ...)."""
    _, y, _ = load_data(dataset)
    in_main_figure = {method_for(row, dataset) for row in ("Classical", "Metric", "Non-metric")}
    methods = [m for m in METHODS if m not in in_main_figure and is_planned(dataset, m)]
    fig, axs = plt.subplots(ncols=len(methods), figsize=(1.7 * len(methods), 1.8), layout="constrained", squeeze=False)
    for ax, method in zip(axs[0], methods):
        Z = load_Z(dataset, method)
        ax.set_aspect("equal", adjustable="datalim")
        ax.scatter(Z[:, 0], Z[:, 1], s=s, ec="none", c=y, cmap=cmap, rasterized=True)
        ax.set_axis_off()
        ax.set_title(METHOD_LABELS.get(method, method))


## Noisy blobs

In [ ]:
embedding_figure("blobs", "blobs", s=2, cmap="tab10", shepard_n=100,
                 shepard_ylim=(0, 2.5), shepard_yticks=(0, 1, 2), circle=True)
other_methods_figure("blobs", s=2, cmap="tab10")

## 1000 genomes

In [ ]:
embedding_figure("genomes", "genomes", s=3, shepard_n=500,
                 shepard_ylim=(0, 3), shepard_yticks=(0, 1, 2, 3), circle=True)
other_methods_figure("genomes", s=3)

## Tasic

In [ ]:
embedding_figure("tasic", "tasic", s=3, shepard_n=500, shepard_xlim=(0, 2.5),
                 shepard_ylim=(0, 6), shepard_yticks=range(7), text_fontsize=6)
other_methods_figure("tasic", s=3)

Shepard diagram of the non-metric embedding, colored by pair of cell classes.

In [ ]:
ann = np.load(paths.DATA / "tasic-data/tasic-ttypes.npy")
types = np.unique(ann)

exc_types = [t for t in types if t[:3] in ["L2/", "L4 ", "L5 ", "L6 ", "L6b"]]
inh_types = [t for t in types if t[:4] in ["Lamp", "Pval", "Serp", "Sncg", "Sst ", "Vip "]]
non_types = [t for t in types if t[:4] in ["Astr", "Endo", "Micr", "OPC ", "Olig", "SMC ", "PVM ", "Peri", "VLMC"]]
print("Uncategorized cell types:", [t for t in types if t not in exc_types + inh_types + non_types])


def tasic_distance_plot(Z, add_isotonic_line=False):
    X, _, _ = load_data("tasic")
    ind = np.random.default_rng(seed=42).choice(X.shape[0], size=500, replace=False)

    d1 = squareform(pdist(X[ind]))
    d2 = squareform(pdist(Z[ind]))
    d1 = d1 / d1.mean()

    fig = plt.figure(figsize=(2.5, 2.5), layout="constrained")
    plt.xlabel("Normalized high-dim. distances")
    plt.ylabel("Embedding distances")

    classes = np.zeros(len(ind)).astype(int) - 10
    classes[np.isin(ann[ind], exc_types)] = 0
    classes[np.isin(ann[ind], inh_types)] = 1
    classes[np.isin(ann[ind], non_types)] = 3
    d_classes = np.abs(classes + classes[:, np.newaxis])

    dd = np.zeros(d_classes.shape).astype(int) - 10
    dd[np.isin(d_classes, [0, 2])] = 0
    dd[d_classes == 1] = 1
    dd[d_classes == 6] = 2
    dd[np.isin(d_classes, [3, 4])] = 3

    labels = ["Exc/exc and inh/inh", "Exc/inh", "Non/non", "Exc/non and inh/non"]
    for i, label in enumerate(labels):
        plt.scatter(d1.ravel()[dd.ravel() == i], d2.ravel()[dd.ravel() == i],
                    s=1, ec="none", label=label, rasterized=True)

    plt.xlim([0, 2.5])
    plt.ylim([0, 6])
    legend = plt.legend()
    for handle in legend.legend_handles:
        handle.set_sizes([50])

    if add_isotonic_line:
        ir = IsotonicRegression(out_of_bounds="clip").fit(d1.ravel(), d2.ravel())
        xgrid = np.arange(0, 2.5, 0.01)
        plt.plot(xgrid, ir.predict(xgrid), "k--")
    return fig


fig = tasic_distance_plot(load_Z("tasic", method_for("Non-metric", "tasic")), add_isotonic_line=True)
save(fig, "tasic-shepard")

## Digits

In [ ]:
embedding_figure("digits", "digits", s=5, cmap="tab10", shepard_n=500,
                 shepard_ylim=(0, 2.5), shepard_yticks=(0, 1, 2))
other_methods_figure("digits", s=5, cmap="tab10")

## MNIST

In [ ]:
embedding_figure("mnist", "mnist", s=.5, cmap="tab10", shepard_n=500,
                 shepard_ylim=(0, 3), shepard_yticks=(0, 1, 2, 3), text_fontsize=6)
other_methods_figure("mnist", s=1, cmap="tab10")